# FraudShield — Entraînement LightGBM

Ce notebook reprend étape par étape le pipeline d'entraînement
avec des explications détaillées de chaque choix technique.

**Sections :**
1. Chargement et Feature Engineering
2. Stratégie de validation croisée (pourquoi OOF ?)
3. Gestion du déséquilibre (scale_pos_weight)
4. Entraînement et courbes d'apprentissage
5. Métriques : AUC-ROC vs PR-AUC
6. Analyse du threshold optimal

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

from src.config import LGBM_PARAMS, N_SPLITS
from src.feature_engineering import run_feature_pipeline

NAVY, GOLD = '#1B2A4A', '#C9A84C'
print('✓ Setup OK')

2026-10-12 21:52:56.269 | INFO     | src.config:<module>:64 - Config FraudShield chargée — DB : fraudshield@localhost


✓ Setup OK


## 1. Chargement (sous-ensemble pour le notebook)

In [2]:
# 100K lignes pour le notebook (le script train.py tourne sur tout le dataset)
X, y = run_feature_pipeline(nrows=100_000, save=False)

print(f"X : {X.shape}")
print(f"Taux fraude : {y.mean()*100:.2f}%")
print(f"Ratio déséquilibre : {(y==0).sum()/(y==1).sum():.1f}")

2026-10-12 21:53:25.538 | INFO     | src.feature_engineering:load_data:54 - Chargement des données IEEE-CIS...
2026-10-12 21:53:33.129 | INFO     | src.feature_engineering:load_data:59 -   transactions : (100000, 394)
2026-10-12 21:53:33.130 | INFO     | src.feature_engineering:load_data:60 -   identity     : (100000, 41)
2026-10-12 21:53:33.381 | SUCCESS  | src.feature_engineering:load_data:63 - ✓ Dataset fusionné : 100,000 lignes × 434 colonnes
2026-10-12 21:53:33.383 | INFO     | src.feature_engineering:load_data:66 -   Taux de fraude : 2.56%
2026-10-12 21:53:33.429 | INFO     | src.feature_engineering:build_features:211 - Feature engineering...
2026-10-12 21:53:33.447 | DEBUG    | src.feature_engineering:add_temporal_features:129 - ✓ Features temporelles ajoutées
2026-10-12 21:53:33.982 | DEBUG    | src.feature_engineering:add_velocity_features:167 - ✓ Features velocity ajoutées
2026-10-12 21:53:34.044 | DEBUG    | src.feature_engineering:add_email_features:183 - ✓ Features email a

X : (100000, 434)
Taux fraude : 2.56%
Ratio déséquilibre : 38.0


## 2. Pourquoi scale_pos_weight ?

Le dataset est déséquilibré : ~2.561% de fraudes seulement.
Sans correction, le modèle apprend à toujours prédire 'légitime' (97.5% accuracy ≠ performance réelle).

**scale_pos_weight** = nb_négatifs / nb_positifs ≈ 38.05

LightGBM multiplie le gradient des fraudes par ce ratio → équilibre l'apprentissage.

In [3]:
n_neg = (y == 0).sum()
n_pos = (y == 1).sum()
spw = n_neg / n_pos

print(f"Légitimes  : {n_neg:,}")
print(f"Fraudes    : {n_pos:,}")
print(f"scale_pos_weight = {spw:.2f}")
print()
print("Interprétation : le modèle pénalise une fraude manquée")
print(f"autant que {spw:.0f} fausses alertes")

Légitimes  : 97,439
Fraudes    : 2,561
scale_pos_weight = 38.05

Interprétation : le modèle pénalise une fraude manquée
autant que 38 fausses alertes


## 3. Entraînement avec 5-fold Stratified CV

In [4]:
params = LGBM_PARAMS.copy()
params['scale_pos_weight'] = spw

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    
    model = lgb.train(
        params,
        dtrain,
        num_boost_round=300,  # réduit pour le notebook
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(100),
        ]
    )
    
    oof_preds[val_idx] = model.predict(X_val)
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    fold_aucs.append(fold_auc)
    print(f'Fold {fold} — AUC = {fold_auc:.4f}')

print(f'\nAUC OOF global : {roc_auc_score(y, oof_preds):.4f}')
print(f'PR-AUC          : {average_precision_score(y, oof_preds):.4f}')

[100]	valid_0's auc: 0.945652
[200]	valid_0's auc: 0.952307
[300]	valid_0's auc: 0.954632
Fold 1 — AUC = 0.9546
[100]	valid_0's auc: 0.934529
[200]	valid_0's auc: 0.940521
[300]	valid_0's auc: 0.943741
Fold 2 — AUC = 0.9439
[100]	valid_0's auc: 0.935578
[200]	valid_0's auc: 0.944294
[300]	valid_0's auc: 0.948285
Fold 3 — AUC = 0.9484
[100]	valid_0's auc: 0.92961
[200]	valid_0's auc: 0.936908
[300]	valid_0's auc: 0.939219
Fold 4 — AUC = 0.9393
[100]	valid_0's auc: 0.937627
[200]	valid_0's auc: 0.945045
[300]	valid_0's auc: 0.947623
Fold 5 — AUC = 0.9479

AUC OOF global : 0.9468
PR-AUC          : 0.7616


## 4. Analyse du threshold optimal

Le threshold par défaut est 0.5, mais sur un problème déséquilibré,
un threshold plus bas peut améliorer le rappel (détecter plus de fraudes)
au prix de plus de fausses alertes.

In [5]:
from sklearn.metrics import precision_recall_curve, f1_score

thresholds = np.arange(0.1, 0.95, 0.05)
results = []

for t in thresholds:
    preds_bin = (oof_preds >= t).astype(int)
    tp = ((preds_bin==1) & (y==1)).sum()
    fp = ((preds_bin==1) & (y==0)).sum()
    fn = ((preds_bin==0) & (y==1)).sum()
    
    prec  = tp / max(tp + fp, 1)
    rec   = tp / max(tp + fn, 1)
    f1    = 2 * prec * rec / max(prec + rec, 1e-9)
    
    results.append({'threshold': t, 'precision': prec, 'recall': rec, 'f1': f1,
                    'n_alerts': preds_bin.sum()})

df_th = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df_th['threshold'], df_th['precision'], color=NAVY, label='Précision', lw=2)
axes[0].plot(df_th['threshold'], df_th['recall'], color=GOLD, label='Rappel', lw=2)
axes[0].plot(df_th['threshold'], df_th['f1'], color='green', label='F1', lw=2, linestyle='--')
axes[0].set_xlabel('Threshold')
axes[0].set_title('Précision / Rappel / F1 vs Threshold')
axes[0].legend()

axes[1].bar(df_th['threshold'], df_th['n_alerts'], width=0.04, color=NAVY, alpha=0.7)
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Nombre d\'alertes')
axes[1].set_title('Volume d\'alertes vs Threshold')

plt.tight_layout()
plt.savefig('../reports/threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

best_f1_row = df_th.loc[df_th['f1'].idxmax()]
print(f"\nThreshold optimal (F1) : {best_f1_row['threshold']:.2f}")
print(f"  Précision : {best_f1_row['precision']:.3f}")
print(f"  Rappel    : {best_f1_row['recall']:.3f}")
print(f"  F1        : {best_f1_row['f1']:.3f}")
print(f"  Alertes   : {best_f1_row['n_alerts']:.0f}")


Threshold optimal (F1) : 0.70
  Précision : 0.816
  Rappel    : 0.667
  F1        : 0.734
  Alertes   : 2092
